# Desktop Helper — SmolLM2 era fine-tuning

Fine-tunes `DesktopHelperLM` (modern Llama-style architecture, SmolLM2-1.7B-Instruct weights transplanted) on Dolly + tool-call data.

**Runtime:** GPU required (Runtime → Change runtime type). Free-tier T4/L4 (~15GB) works at batch 1 + grad-accum; an A100 (Colab Pro) is comfortable.

**Drive space:** each epoch checkpoint is ~3.4GB (bf16, no optimizer state). Clear old OPT-era checkpoints from `desktop_helper_checkpoints/` if space is tight.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo (smollm2-1.7b branch)

In [ ]:
import os
if os.path.exists('/content/Desktop_Helper'):
    %cd /content/Desktop_Helper
    !git pull  # repo already cloned this session — refresh it
else:
    !git clone https://github.com/JaysonSalemmo/Desktop_Helper.git
    %cd Desktop_Helper
    !git checkout smollm2-1.7b


## 2b. GPU check — do not proceed on a CPU runtime

Kai's first attempt ran on a CPU runtime: the transplant was OOM-killed and training refused to start. Runtime → Change runtime type → **T4 GPU** (or better), then re-run from the top.

In [ ]:
import torch
assert torch.cuda.is_available(), 'NO GPU — Runtime → Change runtime type → T4 GPU, then restart & re-run'
print(torch.cuda.get_device_name(0))

## 3. Install dependencies

Colab ships torch + CUDA. bitsandbytes provides the 8-bit optimizer that makes 1.7B fit on a T4.

In [ ]:
!pip install -q datasets tokenizers tensorboard transformers 'bitsandbytes>=0.44'

## 4. Transplant SmolLM2 → DesktopHelperLM (in-notebook)

Downloads ~3.4GB on datacenter bandwidth and writes the fp32 transplant to local disk (not Drive — it's rebuilt each session in ~2 min).

In [ ]:
!python -m model.load_base --output /content/smol_transplant.pt

## 5. Safety gate: logit-equivalence test

The OPT-era lesson: a transplant bug once cost a full training run to discover. This compares our architecture against HuggingFace's reference — **do not train if this fails.**

In [ ]:
!mkdir -p model/checkpoints
!ln -sf /content/smol_transplant.pt model/checkpoints/smol_transplant.pt
!python -m pytest tests/test_transplant.py -q

## 6. Regenerate the tool-call data

Deterministic (same seed as local). Includes the no-tool chat category (routing contrast).

In [ ]:
!python -m model.data.tool_calls --count 4000 --seed 42 --output data/tool_calls.jsonl

## 7. Launch TensorBoard (run before training)

Watch `loss/train` and — new this era — `loss/held_out`: the catastrophic-forgetting alarm. If held-out loss climbs while train loss falls, stop and reduce epochs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1/runs

## 8. Fine-tune

Light-touch adaptation: 3 epochs, single LR 2e-5, pure bf16, 8-bit AdamW, gradient checkpointing. Checkpoints go straight to Drive under `smol_run1/` (each run gets its own folder — no more overwrite trap).

If OOM on a T4: batch-size is already 1; reduce `--grad-accum` won't help memory — close other notebooks or use an A100.

In [ ]:
!python -m model.train \
  --checkpoint /content/smol_transplant.pt \
  --tokenizer model/hf_tokenizer \
  --tool-calls data/tool_calls.jsonl \
  --output /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1/ \
  --epochs 3 \
  --batch-size 1 \
  --grad-accum 32

## 9. Faithfulness eval

Scores RESULT-copying on held-out entities. OPT-era final score was 88% (run 5); the transplant's pretrained embeddings should start far higher.

In [ ]:
!python -m model.eval_faithfulness --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1/epoch_03.pt --tokenizer model/hf_tokenizer

## 10. Chat sanity — the Ojomala Adar memorial cell

The entire point of this era: "Hello" must produce sense.

In [ ]:
for prompt in ["Hello!", "How are you today?", "What can you do?", "Tell me a fun fact about space."]:
    !python -m model.generate --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1/epoch_03.pt --tokenizer model/hf_tokenizer --prompt "{prompt}" --max-new-tokens 60

## 11. Routing warm-start (run after the main fine-tune)

Run-1 result: chat is excellent, held-out loss fell every epoch — but the 11 tool-token
rows (the only from-scratch parameters) stayed too weak to fire (p≈0.02-0.04 at the
routing position → 0% eval, "routed to None").

This phase resumes from epoch_03 and trains **only those 11 embedding rows** —
gradient-masked, weight-decay 0, everything else frozen and bit-identical (unit-tested).
Tool-call data only: forgetting is impossible when nothing else can move.
Fast: only embedding grads flow, ~4k examples, 2 epochs.

In [ ]:
!python -m model.train \
  --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1/epoch_03.pt \
  --tokenizer model/hf_tokenizer \
  --tool-calls data/tool_calls.jsonl \
  --output /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1_warm/ \
  --embeddings-only --no-dolly \
  --lr 1e-3 --epochs 2 \
  --batch-size 2 --grad-accum 16

## 12. Re-eval the warm-started checkpoint

Expect routing to fire now. Chat cannot have changed (those weights were frozen).

In [ ]:
!python -m model.eval_faithfulness --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1_warm/epoch_02.pt --tokenizer model/hf_tokenizer

## 13. Files-token cutover — add `[CALL: files]` to the existing warm checkpoint

The `smol_run1_warm` run trained 11 tool rows at vocab **49163**. A new tool token,
`[CALL: files]` (id **49163**), was appended *after* those, so every learned row keeps
its meaning and this only adds one row (vocab **49164**).

Rather than re-run the multi-hour fine-tune, this resumes from the working warm checkpoint:
1. **Resize** — grow the embedding table 49163 → 49164, seeding the new files row from
   the mean of the existing tool rows (starts in "tool space", not from noise).
2. **Warm-start only row 49163** — `--first-new-row 49163` gradient-masks everything below
   it, so the 11 trained rows and every block stay bit-identical while just the files row
   learns to fire. Fast: one row, tool-call data only, 2 epochs.

The tool-call data (§6) already emits `[CALL: files]` examples, so no data change is needed.
Validated locally: resize preserves all 49163 rows bit-identically and chat is unchanged.

In [ ]:
!python -m model.resize_embeddings \
  --in  /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1_warm/epoch_02.pt \
  --out /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1_files_init.pt

In [ ]:
!python -m model.train \
  --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1_files_init.pt \
  --tokenizer model/hf_tokenizer \
  --tool-calls data/tool_calls.jsonl \
  --output /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1_files/ \
  --embeddings-only --first-new-row 49163 --no-dolly \
  --lr 1e-3 --epochs 2 \
  --batch-size 2 --grad-accum 16

## 14. Re-eval — files routing should now fire

Chat and the other 9 tools cannot have changed (their rows were frozen). Expect
`[CALL: files]` to fire on the two file cases ("Find Kai's resume", "Where's my budget
spreadsheet?") added to the faithfulness set. Then download `smol_run1_files/epoch_02.pt`
and drop it in with `model/hf_tokenizer` for the app cutover.

In [ ]:
!python -m model.eval_faithfulness --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1_files/epoch_02.pt --tokenizer model/hf_tokenizer

## 15. RUN 7 — routing retrain with hard negatives

Run 6 trained ONLY the files row, and live testing showed it over-fires: "Play a song
by Bruno Mars" → files, "When are my reminders set for?" → files, "Add milk to my
shopping list" → files. Root cause: those phrasings were in NO tool's training data,
and with every other row frozen nothing could compete for them. The dispatcher has
stopgap guards; **this run moves the correctness into the weights.**

Baseline (`model/eval_routing.py` on `smol_run1_files/epoch_02.pt`): **19/30 (63%),
7 over-fires, 4 under-fires** — it catches every live failure plus two the app's
router band-aids were silently covering.

What's different in the data (regenerated below):
- **spotify**: play-request family ("Play a song by {artist}", "Put on some jazz", …)
- **reminders**: creates ("Remind me to X", "Add milk to my shopping list") and
  due-focused reads ("When are my reminders set for?") with results matching the
  app's `(due Fri Jul 26 at 5:00 PM)` readback format

Training: resume from run 6's checkpoint, warm-start **ALL tool rows** (no
`--first-new-row` → defaults to the first tool id, 49152+, so the 12 rows can
rebalance against each other). Everything else frozen — forgetting still impossible.
LR 5e-4 (rows are already trained; 1e-3 was for a fresh row).

In [ ]:
!git pull
!python -m model.data.tool_calls --count 4000 --seed 42 --output data/tool_calls.jsonl
!python -m model.train \
  --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1_files/epoch_02.pt \
  --tokenizer model/hf_tokenizer \
  --tool-calls data/tool_calls.jsonl \
  --output /content/drive/MyDrive/desktop_helper_checkpoints/smol_run7_routing/ \
  --embeddings-only --no-dolly \
  --lr 5e-4 --epochs 2 \
  --batch-size 2 --grad-accum 16

## 16. Run-7 evals — routing is the headline number

**Routing** (new): bare model, no routers — the number that was invisible in run 6.
Target: the three live over-fires route correctly AND the files positives
("Find Kai's resume", "Where's my budget spreadsheet?") survive the rebalance.
**Faithfulness**: must hold ~where it was (copying weights were frozen; only the
tool-token embedding rows moved).

If routing improves but a files positive regresses below its run-6 result, try
epoch_01 (less rebalancing) before touching hyperparameters.

In [ ]:
!python -m model.eval_routing --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run7_routing/epoch_02.pt --tokenizer model/hf_tokenizer
!python -m model.eval_faithfulness --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run7_routing/epoch_02.pt --tokenizer model/hf_tokenizer